# Naive Bayes
### Imports

In [184]:
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import export_text
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import matplotlib.pyplot as plt
import numpy as np
from imblearn.over_sampling import SMOTE



#### Load and check data

In [185]:
# Load the training and test data
X_train = pd.read_csv('Datasets/alternative_X_train.csv')
X_test = pd.read_csv('Datasets/alternative_X_test.csv')
y_train = pd.read_csv('Datasets/y_train.csv')
y_test = pd.read_csv('Datasets/y_test.csv')


In [186]:
# Ensure the test data has the same columns as the training data
X_test = X_test[X_train.columns]
y_test = y_test[y_train.columns] 

In [187]:
# check the shape of the data
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

# check collumn names are identical
print("X_train columns:", X_train.columns.tolist())
print("X_test columns:", X_test.columns.tolist())


print(X_test)



Shape of X_train: (404, 217)
Shape of y_train: (404, 10)
Shape of y_test: (101, 10)
X_train columns: ['IMDb rating', 'duration', 'content rating', 'year', 'month', 'genre 1', 'genre 2', 'genre 3', 'writer 1', 'writer 2', 'writer 3', 'star 1', 'star 2', 'star 3', 'director 1', 'director 2', 'director 3', 'title_tfidf_1', 'title_tfidf_2', 'title_tfidf_3', 'title_tfidf_4', 'title_tfidf_5', 'title_tfidf_6', 'title_tfidf_7', 'title_tfidf_8', 'title_tfidf_9', 'title_tfidf_10', 'title_tfidf_11', 'title_tfidf_12', 'title_tfidf_13', 'title_tfidf_14', 'title_tfidf_15', 'title_tfidf_16', 'title_tfidf_17', 'title_tfidf_18', 'title_tfidf_19', 'title_tfidf_20', 'title_tfidf_21', 'title_tfidf_22', 'title_tfidf_23', 'title_tfidf_24', 'title_tfidf_25', 'title_tfidf_26', 'title_tfidf_27', 'title_tfidf_28', 'title_tfidf_29', 'title_tfidf_30', 'title_tfidf_31', 'title_tfidf_32', 'title_tfidf_33', 'title_tfidf_34', 'title_tfidf_35', 'title_tfidf_36', 'title_tfidf_37', 'title_tfidf_38', 'title_tfidf_39', 't

### missing values and NaN

In [188]:
# check if there are missing values or collumns
train_columns = set(X_train.columns)
test_columns = set(X_test.columns)

missing_in_test = train_columns - test_columns
extra_in_test = test_columns - train_columns

print("Missing in X_test:", missing_in_test)
print("Extra in X_test:", extra_in_test)



Missing in X_test: set()
Extra in X_test: set()


In [189]:
# remove NaN values in the data
X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

## Initialize and train the Naive Bayes per collumn

In [199]:
# Initialize a dictionary to store results for each column
results = {}

# Initialize SMOTE for dealing with class imbalance
smote = SMOTE(sampling_strategy=0.5, k_neighbors=9, random_state=42)

# Train and evaluate a model for each target column
for column in y_train.columns:
    print(f"Training for target column: {column}")

    # Extract the current target
    y_train_col = y_train[column].values.squeeze()
    y_test_col = y_test[column].values.squeeze()
    
    """Due to a class imbalance witin the target data, we will use different techniques 
    to try and balance the classes: 
    A regular initialization, adding class weights, or using SMOTE to try and balance the classes.
    """
    # Regular initialization and training of the model
    # nb_model = MultinomialNB()
    # nb_model.fit(X_train, y_train_col)

    # adding class weights to balance the classes
    # class_prior = [0.5, 0.5]  # Equal weight for both classes
    # nb_model = MultinomialNB(class_prior=class_prior)
    # nb_model.fit(X_train, y_train_col)
    
    # Using SMOTE to balance the classes
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train_col)
    nb_model = MultinomialNB()
    nb_model.fit(X_train_resampled, y_train_resampled)

    # Make predictions
    y_pred_col = nb_model.predict(X_test)
    
    # Evaluate the model
    accuracy = accuracy_score(y_test_col, y_pred_col)
    report = classification_report(y_test_col, y_pred_col)
    
    # Store results
    results[column] = {
        "accuracy": accuracy,
        "classification_report": report,
    }
    
    # Print results
    print(f"Accuracy for {column}: {accuracy:.2f}")
    print(report)

Training for target column: alien point of view
Accuracy for alien point of view: 0.92
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        93
           1       0.00      0.00      0.00         8

    accuracy                           0.92       101
   macro avg       0.46      0.50      0.48       101
weighted avg       0.85      0.92      0.88       101

Training for target column: belonging
Accuracy for belonging: 0.89
              precision    recall  f1-score   support

           0       0.89      1.00      0.94        90
           1       0.00      0.00      0.00        11

    accuracy                           0.89       101
   macro avg       0.45      0.50      0.47       101
weighted avg       0.79      0.89      0.84       101

Training for target column: father and son
Accuracy for father and son: 0.21
              precision    recall  f1-score   support

           0       0.78      0.08      0.15        85
      

/Users/reneehagemans/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/reneehagemans/Library/Python/3.9/lib/python/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The SMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/Users/reneehagemans/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Prec

Accuracy for obsession: 0.88
              precision    recall  f1-score   support

           0       0.88      1.00      0.94        89
           1       0.00      0.00      0.00        12

    accuracy                           0.88       101
   macro avg       0.44      0.50      0.47       101
weighted avg       0.78      0.88      0.83       101

Training for target column: romantic love
Accuracy for romantic love: 0.49
              precision    recall  f1-score   support

           0       0.92      0.41      0.57        83
           1       0.23      0.83      0.37        18

    accuracy                           0.49       101
   macro avg       0.58      0.62      0.47       101
weighted avg       0.80      0.49      0.53       101

Training for target column: time travel
Accuracy for time travel: 0.34
              precision    recall  f1-score   support

           0       0.93      0.28      0.43        90
           1       0.12      0.82      0.21        11

    acc

/Users/reneehagemans/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/reneehagemans/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/reneehagemans/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita